# Test: `get_synonyms()` for all APIs_pipe clients

Run from the **project root** so that `scripts/` is on the Python path.

Covers:
- COL, GBIF, GenBank, Index Fungorum, ITIS, MushroomObserver, Tropicos
- FishBase
- Symbiota portals: MyCoPortal, Lichen, Bryophyte, CCH2, SERNEC, NANSH, Macroalgae, Pterido, NE Herbaria, Mid-Atlantic, SW Biodiversity

In [1]:
import time
import sys, os
# Ensure the project root is on sys.path when running from notebooks.apis_pipe/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from IPython.display import display

from scripts.apis_pipe.col import COLAPI
from scripts.apis_pipe.gbif import GBIFAPI
from scripts.apis_pipe.genbank import GenBankAPI
from scripts.apis_pipe.index_fungorum import IndexFungorumAPI
from scripts.apis_pipe.itis import ITISAPI
from scripts.apis_pipe.mushroomobs import MushroomObserverAPI
from scripts.apis_pipe.symbiota import SymbiotaAPI
from scripts.apis_pipe.tropicos import TropicosAPI
from scripts.apis_pipe.fishbase import FishBaseAPI

In [2]:
import pandas as pd

pd.set_option("display.max_colwidth", None)


def show(df):
    """Display a synonym DataFrame with api_link rendered as a clickable anchor."""
    if df.empty:
        display(df)
        return
    display(
        df.style.format(
            {
                "api_link": lambda url: (
                    f'<a href="{url}" target="_blank">{url}</a>'
                    if url and url not in ("U", "")
                    else url
                )
            }
        )
    )

---
## Standard APIs

In [3]:
from scripts.config import (
    COL_PORTAL,
    GBIF_PORTAL,
    INDEX_FUNGORUM_PORTAL,
    ITIS_PORTAL,
    MUSHROOM_OBSERVER_PORTAL,
    TROPICOS_PORTAL,
    FISHBASE_PORTAL
)

STANDARD_QUERIES = [
    # (portal, client, accepted_species, synonym_of_accepted)
    # (COL_PORTAL,               COLAPI(),              "Quercus robur",    "Quercus atrosanguinea"),
    # (GBIF_PORTAL,              GBIFAPI(),             "Amanita muscaria", "Agaricus muscarius"),
    # (INDEX_FUNGORUM_PORTAL,    IndexFungorumAPI(),    "Amanita muscaria", "Agaricus muscarius"),
    # (MUSHROOM_OBSERVER_PORTAL, MushroomObserverAPI(), "Amanita muscaria", "Amanita amerimuscaria"),
    (TROPICOS_PORTAL,          TropicosAPI(),         "Quercus robur",    "Quercus pedunculata"),
    # (FISHBASE_PORTAL,          FishBaseAPI(),         "Gadus morhua",     "Gadus callarias"),
    # (ITIS_PORTAL,              ITISAPI(),             "Oncorhynchus mykiss",    "Salmo mykiss"),
]

In [4]:
# display(GBIFAPI().get_synonyms("Agaricus muscarius"))

In [4]:
for portal, client, accepted, synonym in STANDARD_QUERIES:
    for species, name_type in [(accepted, "accepted"), (synonym, "synonym"), ("Not species", "not species")]:
        print(f"\n{'='*60}")
        print(f"  {portal.display_name}  ({client.__class__.__name__})")
        print(f"  Species:   {species}")
        print(f"  Name type: {name_type}  (synonym of: {accepted})" if name_type == "synonym" else f"  Name type: {name_type}")
        print(f"{'='*60}")
        try:
            result = client.get_synonyms(species)
            print(f"Synonyms returned: {len(result)}")
            show(result)
        except Exception as e:
            print(f"ERROR: {e}")


  Tropicos  (TropicosAPI)
  Species:   Quercus robur
  Name type: accepted
Synonyms returned: 2


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,robur,L.,Sp. Pl. 2: 996,1753,Accepted,N/A,https://www.tropicos.org/name/13100004,13100004
1,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,pedunculata,Ehrh.,N/A,N/A,Synonym,N/A,https://www.tropicos.org/name/13100434,13100434



  Tropicos  (TropicosAPI)
  Species:   Quercus pedunculata
  Name type: synonym  (synonym of: Quercus robur)
Synonyms returned: 2


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,robur,L.,Sp. Pl. 2: 996,1753,Accepted,N/A,https://www.tropicos.org/name/13100004,13100004
1,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,pedunculata,Ehrh.,N/A,N/A,Synonym,N/A,https://www.tropicos.org/name/13100434,13100434



  Tropicos  (TropicosAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


In [6]:
from scripts.config import GENBANK_PORTAL

# GenBank is run separately because NCBI rate-limits unauthenticated callers
# to 3 req/s. Each get_synonyms() makes 2 requests (esearch + efetch), so we
# sleep 1s between calls to stay well under the limit.
GENBANK_QUERIES = [
    ("Amanita muscaria",   "accepted"),
    ("Agaricus muscarius", "synonym"),
    ("Not species",        "not species"),
]
genbank_client = GenBankAPI()
for species, name_type in GENBANK_QUERIES:
    print(f"\n{'='*60}")
    print(f"  {GENBANK_PORTAL.display_name}  (GenBankAPI)")
    print(f"  Species:   {species}")
    print(f"  Name type: {name_type}")
    print(f"{'='*60}")
    try:
        result = genbank_client.get_synonyms(species)
        print(f"Synonyms returned: {len(result)}")
        show(result)
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(1.0)  # NCBI rate limit: max 3 req/s; each call makes 2 requests


  GenBank  (GenBankAPI)
  Species:   Amanita muscaria
  Name type: accepted
_fetch_query_data
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '1', 'retmax': '1', 'retstart': '0', 'idlist': ['41956'], 'translationset': [], 'translationstack': [{'term': 'Amanita muscaria[Scientific Name]', 'field': 'Scientific Name', 'count': '1', 'explode': 'N'}, 'GROUP'], 'querytranslation': 'Amanita muscaria[Scientific Name]'}}
_fetch_synonym_data
<TaxaSet><Taxon>
    <TaxId>41956</TaxId>
    <ScientificName>Amanita muscaria</ScientificName>
    <OtherNames>
        <GenbankCommonName>fly agaric</GenbankCommonName>
        <Synonym>Agaricus muscarius</Synonym>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Agaricus muscarius L., 1753</DispName>
        </Name>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Amanita muscaria (L.) Lam., 1783</DispName>
        </Name>
        <Name>
            <ClassCDE>missp

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GenBank,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956
1,GenBank,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956



  GenBank  (GenBankAPI)
  Species:   Agaricus muscarius
  Name type: synonym
_fetch_query_data
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '1', 'retmax': '1', 'retstart': '0', 'idlist': ['41956'], 'translationset': [], 'translationstack': [{'term': 'Agaricus muscarius[All Names]', 'field': 'All Names', 'count': '1', 'explode': 'N'}, 'GROUP'], 'querytranslation': 'Agaricus muscarius[All Names]'}}
_fetch_synonym_data
<TaxaSet><Taxon>
    <TaxId>41956</TaxId>
    <ScientificName>Amanita muscaria</ScientificName>
    <OtherNames>
        <GenbankCommonName>fly agaric</GenbankCommonName>
        <Synonym>Agaricus muscarius</Synonym>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Agaricus muscarius L., 1753</DispName>
        </Name>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Amanita muscaria (L.) Lam., 1783</DispName>
        </Name>
        <Name>
            <ClassCDE>misspelling</Class

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GenBank,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956
1,GenBank,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956



  GenBank  (GenBankAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## Symbiota Portals

In [3]:
from scripts.config import SYMBIOTA_PORTAL_BY_NAME

_SYMBIOTA_QUERIES = [
    # (portal, accepted_species, synonym_of_accepted)
    (SYMBIOTA_PORTAL_BY_NAME["MyCoPortal"],                       "Agaricus campestris",     "Psalliota villatica"),
    (SYMBIOTA_PORTAL_BY_NAME["Lichen Portal"],                    "Xanthoria parietina",     "Physcia parietina"),
    (SYMBIOTA_PORTAL_BY_NAME["Bryophyte Portal"],                 "Pohlia nutans",           "Webera nutans"),
    (SYMBIOTA_PORTAL_BY_NAME["CCH2"],                             "Heteromeles arbutifolia", "Photinia arbutifolia"),
    (SYMBIOTA_PORTAL_BY_NAME["SERNEC"],                           "Magnolia grandiflora",    "Magnolia foetida"),
    (SYMBIOTA_PORTAL_BY_NAME["NANSH"],                            "Rudbeckia hirta",         "Coreopsis hirta"),
    (SYMBIOTA_PORTAL_BY_NAME["Algae Herbarium Portal"],           "Ulva intestinalis",       "Ilea intestinalis"),
    (SYMBIOTA_PORTAL_BY_NAME["Pterido Portal"],                   "Dryopteris filix-mas",    "Nephrodium filix-mas"),
    (SYMBIOTA_PORTAL_BY_NAME["CNH"],                              "Impatiens capensis",      "Impatiens biflora"),
    (SYMBIOTA_PORTAL_BY_NAME["Mid-Atlantic Herbaria Consortium"], "Quercus rubra",           "Quercus maxima"),
    (SYMBIOTA_PORTAL_BY_NAME["swbiodiversity"],                   "Larrea tridentata",       "Larrea mexicana"),
]

In [4]:
for portal, accepted, synonym in _SYMBIOTA_QUERIES:
    for species, name_type in [(accepted, "accepted"), (synonym, "synonym"), ("Not species", "not species")]:
        print(f"\n{'='*60}")
        print(f"  symbiota / {portal.display_name}")
        print(f"  URL:       {portal.base_url}")
        print(f"  Species:   {species}")
        print(f"  Name type: {name_type}  (synonym of: {accepted})" if name_type == "synonym" else f"  Name type: {name_type}")
        print(f"{'='*60}")
        try:
            client = SymbiotaAPI(portal.display_name)
            result = client.get_synonyms(species)
            print(f"Synonyms returned: {len(result)}")
            show(result)
        except Exception as e:
            print(f"ERROR: {e}")


  symbiota / MyCoPortal
  URL:       https://mycoportal.org/portal
  Species:   Agaricus campestris
  Name type: accepted
[MyCoPortal] 'api/v2/taxonomy/search' returned results for 'Agaricus campestris'.
Synonyms returned: 2


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,MyCoPortal,Fungi,Basidiomycota,Agaricomycetes,Agaricaceae,Agaricales,N/A,Agaricus,campestris,L.,N/A,N/A,Accepted,,https://mycoportal.org/portal/taxa/index.php?tid=214862,214862
1,MyCoPortal,N/A,N/A,N/A,N/A,N/A,N/A,Psalliota,villatica,(Brond.) Bres.],N/A,N/A,Synonym,N/A,https://mycoportal.org/portal/taxa/index.php?taxon=214862,214862



  symbiota / MyCoPortal
  URL:       https://mycoportal.org/portal
  Species:   Psalliota villatica
  Name type: synonym  (synonym of: Agaricus campestris)
[MyCoPortal] 'api/v2/taxonomy/search' returned results for 'Psalliota villatica'.
Synonyms returned: 2


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,MyCoPortal,Fungi,Basidiomycota,Agaricomycetes,Agaricaceae,Agaricales,N/A,Agaricus,campestris,L.,N/A,N/A,Accepted,,https://mycoportal.org/portal/taxa/index.php?tid=214862,214862
1,MyCoPortal,N/A,N/A,N/A,N/A,N/A,N/A,Psalliota,villatica,(Brond.) Bres.],N/A,N/A,Synonym,N/A,https://mycoportal.org/portal/taxa/index.php?taxon=214862,214862



  symbiota / MyCoPortal
  URL:       https://mycoportal.org/portal
  Species:   Not species
  Name type: not species
[MyCoPortal] Search endpoint returned no results for 'Not species'. Attempting autocomplete search.
[MyCoPortal] All searches failed or returned no results for 'Not species'.
Synonyms returned: 0


...

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,swbiodiversity,Plantae,Magnoliophyta,,Zygophyllaceae,Zygophyllales,N/A,Larrea,tridentata,(Sessé & Moc. ex DC.) Coville,N/A,N/A,Accepted,,https://swbiodiversity.org/seinet/taxa/index.php?tid=2671,2671
1,swbiodiversity,N/A,N/A,N/A,N/A,N/A,N/A,Larrea,mexicana,Moric.,N/A,N/A,Synonym,N/A,https://swbiodiversity.org/seinet/taxa/index.php?taxon=2671,2671



  symbiota / swbiodiversity
  URL:       https://swbiodiversity.org/seinet
  Species:   Larrea mexicana
  Name type: synonym  (synonym of: Larrea tridentata)
[swbiodiversity] 'api/v2/taxonomy' returned results for 'Larrea mexicana'.
Synonyms returned: 2


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,swbiodiversity,Plantae,Magnoliophyta,,Zygophyllaceae,Zygophyllales,N/A,Larrea,tridentata,(Sessé & Moc. ex DC.) Coville,N/A,N/A,Accepted,,https://swbiodiversity.org/seinet/taxa/index.php?tid=2671,2671
1,swbiodiversity,N/A,N/A,N/A,N/A,N/A,N/A,Larrea,mexicana,Moric.,N/A,N/A,Synonym,N/A,https://swbiodiversity.org/seinet/taxa/index.php?taxon=2671,2671



  symbiota / swbiodiversity
  URL:       https://swbiodiversity.org/seinet
  Species:   Not species
  Name type: not species
[swbiodiversity] Search endpoint returned no results for 'Not species'. Attempting autocomplete search.
[swbiodiversity] All searches failed or returned no results for 'Not species'.
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
